### Notebook *NB02b – Modelos de Machine Learning sin sentimiento (horizonte 5 días)*  
**Autor:** Jesús Daniel Romeral Cortina

**Objetivo:**
Entrenar y evaluar distintos modelos de machine learning utilizando exclusivamente variables financieras del S&P 500, sin incorporar información de sentimiento, con el fin de establecer un baseline de referencia para la predicción direccional a 5 días.

**Metodología (resumen):**
- Se emplean únicamente variables financieras a partir del dataset `sp500_model.csv`.
- **Split temporal**: train anterior a **2022-01-01** y test posterior.
- Selección de hiperparámetros mediante **GridSearchCV** con validación cruzada temporal (**TimeSeriesSplit**).
- Métrica principal: **ROC_AUC** (además de Acc, B_Acc, Precision, Recall y F1).
- Hiperparámetros guardados en `best_params_ml_5d.json` y métricas en `resultados_ml_5d.csv`.


In [1]:
import pandas as pd
import numpy as np
import json
import os
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler 
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier 
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV 

In [2]:

FAMILIA = "ML" 
HORIZONTE = "5d" 
SENTIMIENTO = "BASE"

In [3]:
SEED = 42
np.random.seed(SEED)


In [4]:
MODEL_PATH   = "../../datos/sp500_model.csv"
OUT_PATH = "../../resultados/resultados_ml_5d.csv"

In [5]:
df = pd.read_csv(MODEL_PATH, parse_dates=["Date"])
df = df.sort_values("Date").set_index("Date")

In [6]:
print("Shape:", df.shape)
print("Rango fechas:", df.index.min(), "-", df.index.max())
df.head(3)



Shape: (2811, 18)
Rango fechas: 2013-01-02 00:00:00 - 2024-03-04 00:00:00


,Close,High,Low,Open,Volume,Return,Target_1d,Return_5d_forward,Target_5d,ret_lag_1,ret_lag_2,ret_lag_3,ret_lag_4,ret_lag_5,ret_ma_5,ret_std_5,ret_ma_10,ret_std_10
Date,,,,,,,,,,,,,,,,,,
2013-01-02,1462.420044,1462.430054,1426.189941,1426.189941,4202600000,0.025403,0,-0.000957,0,0.016942,-0.011050,-0.001218,-0.004787,-0.002440,0.005058,0.015419,0.002286,0.012203
2013-01-03,1459.369995,1465.469971,1455.530029,1462.420044,3829730000,-0.002086,1,0.008737,1,0.025403,0.016942,-0.011050,-0.001218,-0.004787,0.005598,0.015030,0.000928,0.011814
2013-01-04,1466.469971,1467.939941,1458.989990,1459.369995,3424290000,0.004865,0,0.003805,1,-0.002086,0.025403,0.016942,-0.011050,-0.001218,0.006815,0.014580,0.002174,0.011468


**Target (5 dias)**
- *Target_5d* es la etiqueta binaria a predecir.
- Se elimina features *forward* para evitar fuga de información.

In [7]:
Y = df["Target_5d"]
X = df.drop(columns=[
    "Target_1d", 
    "Target_5d", 
    "Return_5d_forward",
    "Close",
    "High",
    "Low",
    "Open",
    "Volume"
])
X.head(3)

,Return,ret_lag_1,ret_lag_2,ret_lag_3,ret_lag_4,ret_lag_5,ret_ma_5,ret_std_5,ret_ma_10,ret_std_10
Date,,,,,,,,,,
2013-01-02,0.025403,0.016942,-0.011050,-0.001218,-0.004787,-0.002440,0.005058,0.015419,0.002286,0.012203
2013-01-03,-0.002086,0.025403,0.016942,-0.011050,-0.001218,-0.004787,0.005598,0.015030,0.000928,0.011814
2013-01-04,0.004865,-0.002086,0.025403,0.016942,-0.011050,-0.001218,0.006815,0.014580,0.002174,0.011468


**Split temporal por fecha (train/test)**

In [8]:
H = 5
train_mask = df.index < "2022-01-01"

X_train_raw = X.loc[train_mask].iloc[:-H]
y_train     = Y.loc[train_mask].iloc[:-H]

X_test_raw  = X.loc[~train_mask]
y_test      = Y.loc[~train_mask]

In [9]:
print("Train:", X_train_raw.shape, "Test:",X_test_raw.shape)

pos_train = y_train.sum()
pos_test = y_test.sum()
print("Porcentaje positivos en train:", round(pos_train/ len(y_train) * 100, 1),"%")
print("Porcentaje positivos en test:", round(pos_test / len(y_test) * 100, 1),"%")



Train: (2262, 10) Test: (544, 10)
Porcentaje positivos en train: 62.2 %
Porcentaje positivos en test: 55.7 %


In [10]:
def evaluar_modelo(y_test, y_pred, y_proba, nombre):
    metrics = {
        "Familia": FAMILIA,
        "Modelo": nombre,
        "Horizonte": HORIZONTE,
        "Sentimiento" : SENTIMIENTO,
        "Acc": accuracy_score(y_test, y_pred),
        "B_Acc": balanced_accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_proba),
        "Conf_Matrix": confusion_matrix(y_test, y_pred),
    }
    return metrics

resultados = []

**Validación cruzada temporal**
- *TimeSeriesSplit* respeta el orden temporal en cada fold.
- Se usa dentro de *GridSearchCV* para elegir hiperparámetros sin mirar el futuro.

In [11]:
tscv = TimeSeriesSplit(n_splits=5, gap=H)

**Baseline (Dummy)**

In [12]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_raw, y_train)
y_pred = dummy.predict(X_test_raw)
y_proba = dummy.predict_proba(X_test_raw)[:, 1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Dummy_MostFreq"))

**Logistic Regression (Pipeline)**
- *StandardScaler* dentro de *Pipeline*.
- grid reducido sobre C y solver

In [13]:

pipe_lr = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

param_grid_lr ={
    "clf__C": [0.01, 0.1, 1, 10],
    "clf__solver": ["liblinear", "lbfgs"],
}

grid_lr = GridSearchCV(
    estimator=pipe_lr,
    param_grid=param_grid_lr,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1,
)


grid_lr.fit(X_train_raw, y_train)

best_params = grid_lr.best_params_
LR_PARAMS_5D = {k.replace("clf__", ""): v for k, v in best_params.items()}


y_pred = grid_lr.predict(X_test_raw)
y_proba = grid_lr.predict_proba(X_test_raw)[:, 1]

resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Logistic_Reg"))



**Random Forest**

- grid reducido sobre max_depth, min_samples_leaf y nº de árboles.

In [14]:
param_grid_rf = {
    "n_estimators": [100, 300],
    "max_depth": [3, 5, 10],
    "min_samples_leaf": [5, 20, 50],
}


rf = RandomForestClassifier(
    max_features='sqrt',
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)

grid_rf = GridSearchCV(rf, param_grid_rf, cv=tscv, scoring='roc_auc', n_jobs=-1)
grid_rf.fit(X_train_raw, y_train)

RF_PARAMS_5D = grid_rf.best_params_


y_pred = grid_rf.predict(X_test_raw)
y_proba = grid_rf.predict_proba(X_test_raw)[:, 1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Random_Forest"))

**HistGradientBoosting (Scikit-Learn native)**
- grid reducido sobre learning_rate, max_depth, min_samples_leaf.

In [15]:
param_grid_hgb = {
    "learning_rate": [0.03, 0.1],
    "max_depth": [3, 5, 10],
    "min_samples_leaf": [10, 20, 50],
}

hgb = HistGradientBoostingClassifier(random_state=SEED)


grid_hgb = GridSearchCV(hgb, param_grid_hgb, cv=tscv, scoring='roc_auc', n_jobs=-1)
grid_hgb.fit(X_train_raw, y_train)
 
HGB_PARAMS_5D = grid_hgb.best_params_


y_pred = grid_hgb.predict(X_test_raw)
y_proba = grid_hgb.predict_proba(X_test_raw)[:, 1]
resultados.append(evaluar_modelo(y_test, y_pred, y_proba, "Hist_GB"))

**Guardado de hiperparámetros**
- Se exportan a JSON para reutilizarlos en notebooks posteriores.

In [16]:
BEST_PARAMS_5D = {
    "LogisticRegression": LR_PARAMS_5D,
    "RandomForest": RF_PARAMS_5D,
    "HistGradientBoosting": HGB_PARAMS_5D
}

with open("../../resultados/best_params_ml_5d.json","w") as f:
    json.dump(BEST_PARAMS_5D, f, indent=4)

print("Hiperparámetros fijados (ML/5d):")
print(json.dumps(BEST_PARAMS_5D, indent=4))


Hiperparámetros fijados (ML/5d):
{
    "LogisticRegression": {
        "C": 0.01,
        "solver": "liblinear"
    },
    "RandomForest": {
        "max_depth": 3,
        "min_samples_leaf": 50,
        "n_estimators": 300
    },
    "HistGradientBoosting": {
        "learning_rate": 0.1,
        "max_depth": 10,
        "min_samples_leaf": 50
    }
}


**Guardado de resultados**
- Exportación a CSV para combinar en Notebook de resultados `NB05a_Resultados.ipynb`


In [17]:
os.makedirs("../../resultados", exist_ok=True)
df_res = pd.DataFrame(resultados)
df_res["Conf_Matrix"] = df_res["Conf_Matrix"].astype(str).str.replace("\n", " ", regex=False)
df_res.to_csv(OUT_PATH, index=False)

if os.path.exists(OUT_PATH): 
    print(f"Archivo guardado correctamente en {OUT_PATH}") 
else: 
    print("Error: el archivo no se ha guardado.")
    
print(df_res)


Archivo guardado correctamente en ../../resultados/resultados_ml_5d.csv
  Familia          Modelo Horizonte Sentimiento       Acc     B_Acc  \
0      ML  Dummy_MostFreq        5d        BASE  0.556985  0.500000   
1      ML    Logistic_Reg        5d        BASE  0.527574  0.521569   
2      ML   Random_Forest        5d        BASE  0.516544  0.473036   
3      ML         Hist_GB        5d        BASE  0.545956  0.507504   

   Precision    Recall        F1   ROC_AUC             Conf_Matrix  
0   0.556985  1.000000  0.715466  0.500000  [[  0 241]  [  0 303]]  
1   0.576159  0.574257  0.575207  0.519823  [[113 128]  [129 174]]  
2   0.541841  0.854785  0.663252  0.464415  [[ 22 219]  [ 44 259]]  
3   0.561404  0.844884  0.674572  0.508182  [[ 41 200]  [ 47 256]]  
